##Joining and Analyzing customers and orders

In [0]:
orders_df = spark.read.format('csv').option('header','true').option('inferSchema','true').load('/Volumes/olist_spark_workspace/default/mysamplevolume/orders.csv')
orders_df


DataFrame[order_id: int, customer_id: int, order_date: date, total_amount: double, status: string]

In [0]:
orders_df.show(5)

+--------+-----------+----------+-----------------+---------+
|order_id|customer_id|order_date|     total_amount|   status|
+--------+-----------+----------+-----------------+---------+
|       0|       3692|2024-09-03|547.7160076008001|  Shipped|
|       1|      11055|2024-08-10|577.8942599188381|  Pending|
|       2|       6963|2024-08-22|484.2085562764487|  Pending|
|       3|      13268|2024-09-01|366.3286882431848|Cancelled|
|       4|       1131|2024-08-09|896.9588380686909|  Pending|
+--------+-----------+----------+-----------------+---------+
only showing top 5 rows


In [0]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- status: string (nullable = true)



#Analysis on Orders

In [0]:
orders_df.select('customer_id','order_date','total_amount').show(5)


+-----------+----------+-----------------+
|customer_id|order_date|     total_amount|
+-----------+----------+-----------------+
|       3692|2024-09-03|547.7160076008001|
|      11055|2024-08-10|577.8942599188381|
|       6963|2024-08-22|484.2085562764487|
|      13268|2024-09-01|366.3286882431848|
|       1131|2024-08-09|896.9588380686909|
+-----------+----------+-----------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, upper, lit

orders_df.select('customer_id','status').where(upper(col("status")) == lit("SHIPPED")).show(5)


+-----------+-------+
|customer_id| status|
+-----------+-------+
|       3692|Shipped|
|      15211|Shipped|
|       3209|Shipped|
|       8917|Shipped|
|       3516|Shipped|
+-----------+-------+
only showing top 5 rows


In [0]:
## better way

orders_df.filter(upper(col('status')) == 'SHIPPED').show(5)

+--------+-----------+----------+------------------+-------+
|order_id|customer_id|order_date|      total_amount| status|
+--------+-----------+----------+------------------+-------+
|       0|       3692|2024-09-03| 547.7160076008001|Shipped|
|       5|      15211|2024-05-03|486.30584827618145|Shipped|
|       6|       3209|2024-08-24| 800.9795667933956|Shipped|
|       9|       8917|2024-09-01| 878.8901928499223|Shipped|
|      23|       3516|2024-06-15| 668.0685604928779|Shipped|
+--------+-----------+----------+------------------+-------+
only showing top 5 rows


In [0]:
#total order per status

orders_df.groupBy('status').count().show()

+---------+-----+
|   status|count|
+---------+-----+
|  Shipped| 4386|
|Cancelled| 4469|
|Delivered| 4341|
|  Pending| 4457|
+---------+-----+



In [0]:
from pyspark.sql.functions import sum
## total revenue per status
orders_df.groupBy('status').agg(
    sum('total_amount').alias('total_revenue')
).show()

+---------+------------------+
|   status|     total_revenue|
+---------+------------------+
|  Shipped|  2188888.99656208|
|Cancelled|2237958.0091099176|
|Delivered|2210383.5320680086|
|  Pending| 2246861.628608617|
+---------+------------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

#Top 5 Customers
window_spec = Window.orderBy(col('total_amount').desc())

orders_df.withColumn('rank',rank().over(window_spec)).show(5)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-----------+----------+-----------------+---------+----+
|order_id|customer_id|order_date|     total_amount|   status|rank|
+--------+-----------+----------+-----------------+---------+----+
|    5116|      13643|2024-08-03|999.9762145017747|  Pending|   1|
|    7859|      11854|2024-10-14| 999.864258397557|Delivered|   2|
|   12658|      16295|2024-12-02|999.7776744500129|  Pending|   3|
|    7348|       4233|2024-11-12|999.6328650710889|Cancelled|   4|
|    3951|      16341|2024-11-30|999.5960583051472|Delivered|   5|
+--------+-----------+----------+-----------------+---------+----+
only showing top 5 rows


In [0]:
#average order value

from pyspark.sql.functions import avg
orders_df.agg(
    avg(col('total_amount')).alias('avg_total_amount')
).show(5)

+------------------+
|  avg_total_amount|
+------------------+
|503.26245773231864|
+------------------+



In [0]:
#monthly order trend
from pyspark.sql.functions import month
orders_df.withColumn('month', month(col('order_date'))).groupBy('month').count().show(5)


+-----+-----+
|month|count|
+-----+-----+
|   12| 1443|
|    1| 1499|
|    6| 1455|
|    3| 1539|
|    5| 1518|
+-----+-----+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import when, count, col, sum, upper
#cancellation percentage
orders_df.agg(
    (sum(when(upper(col('status'))=='CANCELLED',1).otherwise(0))/count("*")  *100).alias("cancel_percentage")
).show()

+------------------+
| cancel_percentage|
+------------------+
|25.315810343850902|
+------------------+



In [0]:
#pivot

df_with_null = orders_df.groupBy('status').pivot('status').count().show(5)

df_with_null = orders_df.groupBy('status').pivot('status').count().fillna(0)
df_with_null.show(5)

+---------+---------+---------+-------+-------+
|   status|Cancelled|Delivered|Pending|Shipped|
+---------+---------+---------+-------+-------+
|  Shipped|     NULL|     NULL|   NULL|   4386|
|Cancelled|     4469|     NULL|   NULL|   NULL|
|Delivered|     NULL|     4341|   NULL|   NULL|
|  Pending|     NULL|     NULL|   4457|   NULL|
+---------+---------+---------+-------+-------+

+---------+---------+---------+-------+-------+
|   status|Cancelled|Delivered|Pending|Shipped|
+---------+---------+---------+-------+-------+
|  Shipped|        0|        0|      0|   4386|
|Cancelled|     4469|        0|      0|      0|
|Delivered|        0|     4341|      0|      0|
|  Pending|        0|        0|   4457|      0|
+---------+---------+---------+-------+-------+



In [0]:
#pivot
orders_df.groupby(
    month(col("order_date")).alias('month')
).pivot('status').count().show(5)

+-----+---------+---------+-------+-------+
|month|Cancelled|Delivered|Pending|Shipped|
+-----+---------+---------+-------+-------+
|   12|      339|      361|    383|    360|
|    1|      369|      379|    383|    368|
|    6|      374|      350|    385|    346|
|    3|      395|      361|    399|    384|
|    5|      375|      405|    354|    384|
+-----+---------+---------+-------+-------+
only showing top 5 rows


In [0]:
##Duplicate customers
orders_df.groupBy('customer_id')\
    .count()\
        .filter(col('count')>1)\
            .show(5)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       7253|    3|
|       6654|    3|
|       6357|    2|
|       6466|    3|
|        496|    3|
+-----------+-----+
only showing top 5 rows


## Analysis on Orders 2